# 📡 IBM Telco Customer Churn Prediction
## End-to-End ML Training Notebook
**Dataset:** IBM Telco Churn (7,032 rows × 21 raw features → 48 engineered)  
**Model:** XGBoost + Threshold Optimization | **Target:** Accuracy ~79% | **ROC-AUC:** 0.82

> **Run:** `Kernel → Restart & Run All`

---
## 0️⃣ Setup & Imports

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

plt.style.use('dark_background')
sns.set_palette('husl')

PROJECT_ROOT = os.path.dirname(os.getcwd())
DATA_PATH  = os.path.join(PROJECT_ROOT, 'data', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
MODEL_PATH = os.path.join(PROJECT_ROOT, 'models', 'churn_pipeline.pkl')
COLS_PATH  = os.path.join(PROJECT_ROOT, 'models', 'feature_columns.pkl')
os.makedirs(os.path.join(PROJECT_ROOT, 'models'), exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
print('Data exists:', os.path.exists(DATA_PATH))

---
## 1️⃣ Load Raw Data

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
print('=== Churn Distribution ===')
print(df_raw['Churn'].value_counts())
print(f"Churn rate: {(df_raw['Churn']=='Yes').mean()*100:.1f}%")
print('\n=== Data Types ==='); print(df_raw.dtypes)

---
## 2️⃣ Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
churn_counts = df_raw['Churn'].value_counts()
colors = ['#36d399', '#fc4f4f']

axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='none')
axes[0].set_title('Churn Count', color='#0ff0fc', fontsize=13)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v+50, f'{v:,}', ha='center', color='white')

axes[1].pie(churn_counts.values, labels=churn_counts.index,
            colors=colors, autopct='%1.1f%%', textprops={'color':'white'},
            wedgeprops={'edgecolor':'#0d1220','linewidth':2})
axes[1].set_title('Churn Rate', color='#0ff0fc', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Key numeric features distributions by churn
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    df_plot = df_raw.copy()
    df_plot[col] = pd.to_numeric(df_plot[col], errors='coerce')
    for label, color in [('No','#36d399'),('Yes','#fc4f4f')]:
        data = df_plot[df_plot['Churn']==label][col].dropna()
        ax.hist(data, bins=40, alpha=0.6, color=color, label=f'Churn={label}', density=True)
    ax.set_title(col, color='#0ff0fc')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Churn by Contract type
ct = pd.crosstab(df_raw['Contract'], df_raw['Churn'], normalize='index') * 100
ct.plot(kind='bar', color=['#36d399','#fc4f4f'], figsize=(8,4), edgecolor='none')
plt.title('Churn Rate by Contract Type', color='#0ff0fc')
plt.xticks(rotation=0)
plt.ylabel('Percentage %')
plt.tight_layout()
plt.show()

---
## 3️⃣ Preprocessing & Feature Engineering

In [ ]:
from src.preprocess import preprocess
X, y = preprocess(DATA_PATH)
print(f'X shape: {X.shape}')
print(f'y distribution: {y.value_counts().to_dict()}')
print(f'Engineered features: num_services, avg_monthly_spend, is_month_to_month, tenure_group')

---
## 4️⃣ Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Churn: {y_train.mean()*100:.1f}%')
print(f'Test : {X_test.shape}  | Churn: {y_test.mean()*100:.1f}%')

---
## 5️⃣ Build & Train XGBoost Pipeline

In [ ]:
NUMERIC_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'num_services']

preprocessor = ColumnTransformer(
    [('num', StandardScaler(), NUMERIC_FEATURES)], remainder='passthrough'
)
xgb_clf = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=1.0,
    min_child_weight=3, reg_alpha=0.05, reg_lambda=1.0,
    eval_metric='auc', random_state=42, use_label_encoder=False
)
pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', xgb_clf)])

print('Training...')
pipeline.fit(X_train, y_train)
print('Done!')

---
## 6️⃣ Evaluate + Threshold Optimisation

In [ ]:
import numpy as np
y_prob = pipeline.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)

# Threshold sweep
best_acc, best_thresh = 0, 0.5
for t in np.arange(0.30, 0.80, 0.01):
    a = accuracy_score(y_test, (y_prob >= t).astype(int))
    if a > best_acc:
        best_acc, best_thresh = a, t

y_pred = (y_prob >= best_thresh).astype(int)
print('=' * 55)
print(f'  Test Accuracy : {best_acc*100:.2f}%  (threshold={best_thresh:.2f})')
print(f'  Test ROC-AUC  : {roc_auc:.4f}')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['No Churn','Churn']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Churn','Churn']).plot(
    ax=axes[0], cmap='Blues', colorbar=True)
axes[0].set_title(f'Confusion Matrix — Acc={best_acc*100:.1f}%', color='#0ff0fc')

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#0ff0fc', linewidth=2.5, label=f'AUC={roc_auc:.4f}')
axes[1].plot([0,1],[0,1], color='#4a5568', linestyle='--')
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#0ff0fc')
axes[1].set_title('ROC Curve', color='#0ff0fc')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importances
xgb_model = pipeline.named_steps['classifier']
fi_df = pd.DataFrame({'feature': X.columns, 'importance': xgb_model.feature_importances_})
fi_df = fi_df.sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(fi_df['feature'], fi_df['importance'],
        color=plt.cm.YlOrRd(fi_df['importance']/fi_df['importance'].max()), edgecolor='none')
ax.set_title('Top 20 Feature Importances', color='#0ff0fc', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7️⃣ 5-Fold Cross-Validation (Overfitting Check)

In [ ]:
print('Running 5-fold CV...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
cv_auc = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

train_acc = accuracy_score(y_train, pipeline.predict(X_train))

print('\n' + '='*55)
print('  5-FOLD CROSS-VALIDATION')
print('='*55)
print(f'  CV Accuracy  : {[round(a*100,2) for a in cv_acc]}')
print(f'  Mean CV Acc  : {cv_acc.mean()*100:.2f}% ± {cv_acc.std()*100:.2f}%')
print(f'  CV ROC-AUC   : {[round(a,4) for a in cv_auc]}')
print(f'  Mean CV AUC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')
print(f'  Train Acc    : {train_acc*100:.2f}%')
print(f'  Train-CV Gap : {(train_acc - cv_acc.mean())*100:.2f}%  (< 3% = no overfit)')
print('='*55)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
folds = [f'Fold {i+1}' for i in range(5)]
axes[0].bar(folds, cv_acc*100, color='#0ff0fc', edgecolor='none', alpha=0.8)
axes[0].axhline(cv_acc.mean()*100, color='#fc4f4f', linestyle='--',
                label=f'Mean={cv_acc.mean()*100:.2f}%')
axes[0].set_title('CV Accuracy per Fold', color='#0ff0fc')
axes[0].set_ylim([70, 90]); axes[0].legend()

axes[1].bar(folds, cv_auc, color='#36d399', edgecolor='none', alpha=0.8)
axes[1].axhline(cv_auc.mean(), color='#fc4f4f', linestyle='--',
                label=f'Mean={cv_auc.mean():.4f}')
axes[1].set_title('CV ROC-AUC per Fold', color='#0ff0fc')
axes[1].set_ylim([0.70, 0.95]); axes[1].legend()
plt.tight_layout(); plt.show()

---
## 8️⃣ Save Model Artifacts

In [ ]:
artifact = {
    'pipeline':   pipeline,
    'threshold':  best_thresh,
    'model_name': 'XGBoost_IBMTelco',
    'accuracy':   best_acc,
    'roc_auc':    roc_auc,
}
joblib.dump(artifact, MODEL_PATH)
joblib.dump(list(X.columns), COLS_PATH)

print('='*55)
print('  FINAL SUMMARY')
print('='*55)
print(f'  Test  Accuracy   : {best_acc*100:.2f}%  (threshold={best_thresh:.2f})')
print(f'  Test  ROC-AUC    : {roc_auc:.4f}')
print(f'  CV    Accuracy   : {cv_acc.mean()*100:.2f}% ± {cv_acc.std()*100:.2f}%')
print(f'  CV    ROC-AUC    : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')
print(f'  Train-CV Gap     : {(train_acc-cv_acc.mean())*100:.2f}%')
print(f'  Model saved      : {os.path.getsize(MODEL_PATH)//1024} KB')
print(f'  Features saved   : {len(X.columns)}')
print('='*55)
print('  Model ready for API deployment!')